# Tuần 3 · Đánh giá cuối và phân tích lỗi
Phiên bản 2.2. Tải vào thư mục làm việc của notebook hai tệp `MSSV_lstm.pt` và
`MSSV_decision.json` do tuần 2 tạo. Giữ checkpoint của chính mình. Nếu chạy cả ba notebook
trong cùng thư mục cục bộ, hai tệp đã có sau khi chạy tuần 2.

Dữ liệu: **Vietnamese AI Feedback OOD 3125**, giấy phép CC BY 4.0. Dùng các tệp CSV đã chia sẵn từ gói lớp. Xem README để đặt tệp và chạy. Tuần 1–2 chỉ đọc train và validation. Nhận tệp test của lớp vào tuần 3.

In [ ]:
import hashlib
import io
import json
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

LABELS = ["negative", "neutral", "positive"]
DATA_VERSION = "ood3125-de7a66b-group42-v1"
CHECKSUMS = {'train': '17e2029dcf20ba77ef95b62b77e4a0458eabea1d36cb72e0794f790340ea1201', 'validation': 'c8148193a9d14963fbc25837f5bb6277bb907d36d1811d2cc25ea6e935847d0c', 'test': '4bee1b48b6dc277742f4d9591c602a720d4a483646e57786df40bb47f2cb7ef7'}


def text_tokens(text):
    """Tokenize word-like spans after Unicode normalization and case folding."""
    return re.findall(r"\w+", unicodedata.normalize("NFC", text).casefold())


def read_split(split):
    filename = f"ood3125_{split}.csv"
    path = Path("data") / filename
    if not path.is_file():
        path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f"Đặt {filename} trong data/ hoặc thư mục làm việc.")
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != CHECKSUMS[split]:
        raise ValueError(f"{filename} không đúng bản chia của lớp DCT1243.")
    frame = pd.read_csv(io.BytesIO(content))
    frame = frame.rename(columns={"sentence": "text"})
    frame["label"] = frame["label"].map(dict(enumerate(LABELS)))
    return frame


def scores(gold, predicted):
    per_label = f1_score(gold, predicted, labels=LABELS, average=None, zero_division=0)
    return {"accuracy": accuracy_score(gold, predicted), "macro_f1": per_label.mean(),
            **{f"f1_{label}": value for label, value in zip(LABELS, per_label)}}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


def fit_baseline(train):
    model = make_pipeline(
        TfidfVectorizer(tokenizer=text_tokens, token_pattern=None, lowercase=False,
                        ngram_range=(1, 2),
                        min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    )
    return model.fit(train["text"], train["label"])

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

MAX_TOKENS = 50
HIDDEN_DIM = 64
MAX_EPOCHS = 6
torch.set_num_threads(2)


def encode(texts, vocabulary):
    rows, lengths = [], []
    for text in texts:
        ids = [vocabulary.get(token, 1) for token in text_tokens(text)[:MAX_TOKENS]]
        lengths.append(max(1, len(ids)))
        rows.append(ids + [0] * (MAX_TOKENS - len(ids)))
    return torch.tensor(rows, dtype=torch.long), torch.tensor(lengths)


class SentimentLSTM(nn.Module):
    def __init__(self, vocabulary_size, initial_embeddings=None):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, EMBED_DIM, padding_idx=0)
        if initial_embeddings is not None:
            with torch.no_grad():
                self.embedding.weight.copy_(initial_embeddings)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(HIDDEN_DIM, len(LABELS))

    def forward(self, token_ids, lengths):
        embeddings = self.embedding(token_ids)  # [batch, time, embedding]
        packed = pack_padded_sequence(embeddings, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)  # [1, batch, hidden]
        return self.classifier(self.dropout(hidden[-1]))  # [batch, 3]


def predict_lstm(model, texts, vocabulary):
    model.eval()
    token_ids, lengths = encode(texts, vocabulary)
    batches = DataLoader(TensorDataset(token_ids, lengths), batch_size=128)
    with torch.no_grad():
        indices = torch.cat([model(ids, lens).argmax(1) for ids, lens in batches])
    return np.array(LABELS)[indices.numpy()]

def checkpoint_digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

EMBED_DIM = 100

In [ ]:
STUDENT_ID = ""
if not STUDENT_ID.isascii() or not STUDENT_ID.isdigit() or len(STUDENT_ID) < 3:
    raise ValueError("Điền MSSV gồm ít nhất ba chữ số rồi chạy lại ô này.")
PERSONAL_SEED = int(STUDENT_ID[-3:])
print("MSSV:", STUDENT_ID, "| Seed lấy mẫu:", PERSONAL_SEED)

In [ ]:
decision = json.loads(Path(f"{STUDENT_ID}_decision.json").read_text(encoding="utf-8"))
checkpoint = Path(f"{STUDENT_ID}_lstm.pt")
if (decision.get("dataset") != DATA_VERSION
        or decision["student_id"] != STUDENT_ID or decision["model"] not in {"E3", "E4"}
        or decision["epoch"] not in range(1, MAX_EPOCHS + 1)
        or checkpoint_digest(checkpoint) != decision["checkpoint_sha256"]):
    raise ValueError("Dùng đúng cặp tệp được tạo ở cuối notebook tuần 2.")
saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
vocabulary = saved["vocabulary"]
lstm = SentimentLSTM(len(vocabulary))
lstm.load_state_dict(saved["weights"])
print("QUYET_DINH", json.dumps(decision))

## C4 · Đánh giá E1 và LSTM đã khóa
Đọc test sau khi kiểm tra checkpoint. Ma trận nhầm lẫn có hàng là nhãn thật và cột là nhãn dự đoán.
Báo cáo Accuracy, macro-F1 và F1 từng nhãn. Nêu một khác biệt giữa LSTM và E1 trên test, có số liệu. Xác định nhãn có F1 thấp nhất của LSTM (ghi cả hai nếu bằng nhau) và đối chiếu với ma trận nhầm lẫn.
Đọc chênh lệch macro-F1 validation–test của LSTM từ kết quả và nêu một giới hạn của so sánh.

In [ ]:
train = read_split("train")
baseline = fit_baseline(train)
test = read_split("test")
baseline_predictions = baseline.predict(test["text"])
predictions = predict_lstm(lstm, test["text"], vocabulary)
test_scores = pd.DataFrame({
    "E1": scores(test["label"], baseline_predictions),
    decision["model"]: scores(test["label"], predictions),
}).T
display(test_scores.round(4))
print("Macro-F1 validation → test:", round(decision["validation_macro_f1"], 4),
      "→", round(test_scores.loc[decision["model"], "macro_f1"], 4))
matrix = confusion_matrix(test["label"], predictions, labels=LABELS)
display(pd.DataFrame(matrix, index=LABELS, columns=LABELS)
        .rename_axis(index="Nhãn thật", columns="Nhãn dự đoán"))

### Phân tích lỗi
Dùng seed cá nhân để lấy tối đa một câu dự đoán sai cho mỗi nhãn thật. Nếu một nhãn không có lỗi,
ghi số lỗi bằng 0. Với từng câu lấy được, ghi mã dòng, nhãn thật, nhãn dự đoán và chi tiết trong câu
hỗ trợ cách diễn giải. Nêu nguyên nhân dưới dạng giả thuyết. Giữ cấu hình đã khóa trong suốt phần này.

In [ ]:
errors = test.assign(predicted=predictions).loc[lambda frame: frame["label"] != frame["predicted"]]
for label in LABELS:
    candidates = errors[errors["label"] == label]
    print(label, "| số lỗi:", len(candidates))
    if not candidates.empty:
        display(candidates.sample(n=1, random_state=PERSONAL_SEED).reset_index(drop=True))

## Tự chọn · Thử một cặp câu tự viết
Tạo hai câu ngắn chỉ khác một yếu tố, chẳng hạn dấu hiệu phủ định. Ghi nhãn kỳ vọng trước khi chạy,
dự đoán bằng LSTM đã khóa rồi đối chiếu. Dùng kết quả để phân tích riêng cặp câu này.

In [ ]:
examples = []  # Optional: Add the two sentences after recording the expected labels.
if examples:
    display(pd.DataFrame({"text": examples,
                          "predicted": predict_lstm(lstm, examples, vocabulary)}))